In [5]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  EXPERIMENT 6 GENERATION CODE                                  ║
# ║  Dataset: Vietnam Online Privacy Survey                                  ║

# ║  E6: CTGAN + Segmentation  (SDV CTGANSynthesizer per segment)            ║
# ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# Install:
#   pip install sdv            # for E6
!pip install sdv

import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder

SEED    = 42
np.random.seed(SEED)
N_SYNTH = 467

# ── SHARED SETUP ──────────────────────────────────────────────────────────
real = pd.read_excel("REAL_DATA_PRIVACY.xlsx")
cols = list(real.columns)

encoders = {}
real_enc = pd.DataFrame()
for col in cols:
    le = LabelEncoder()
    le.fit(real[col].astype(str))
    encoders[col] = le
    real_enc[col] = le.transform(real[col].astype(str))

km = KMeans(n_clusters=5, random_state=SEED, n_init=50)
labels = km.fit_predict(real_enc.values)
real["cluster"] = labels

shares = {c:(labels==c).sum()/len(labels) for c in range(5)}
n_per  = {}; rem = N_SYNTH
for c in range(4):
    n_per[c] = int(round(shares[c]*N_SYNTH)); rem -= n_per[c]
n_per[4] = rem

print(f"Real data: {real.shape} | n_per: {n_per}")


def decode_df(df_enc):
    out = pd.DataFrame()
    for col in cols:
        vals = np.round(df_enc[col].values).astype(int)
        vals = np.clip(vals, 0, len(encoders[col].classes_) - 1)
        out[col] = encoders[col].inverse_transform(vals)
    return out




# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  EXPERIMENT 6 — CTGAN + SEGMENTATION (SDV)                              ║
# ╚══════════════════════════════════════════════════════════════════════════╝
def generate_e6():
    from sdv.single_table import CTGANSynthesizer
    from sdv.metadata import SingleTableMetadata

    pieces = []
    for c in range(5):
        sub = real[real["cluster"] == c][cols].reset_index(drop=True)
        print(f"E6 | Segment {c}: fitting CTGAN on n={len(sub)}...")

        metadata = SingleTableMetadata()
        metadata.detect_from_dataframe(sub)
        for col in sub.columns:
            if col == 'No': # Assuming 'No' is an identifier column
                metadata.update_column(column_name=col, sdtype='id')
            else:
                metadata.update_column(column_name=col, sdtype="categorical")

        syn = CTGANSynthesizer(
            metadata,
            epochs=500,
            batch_size=50,   # small batch for small cluster sizes
            verbose=False,
        )
        syn.fit(sub)
        chunk = syn.sample(num_rows=n_per[c])
        pieces.append(chunk)
        print(f"   Sampled {n_per[c]} rows.")

    synth = (pd.concat(pieces, ignore_index=True)
               .sample(frac=1, random_state=SEED)
               .reset_index(drop=True))
    synth.to_excel("exp6_ctgan_segmentation.xlsx", index=False)
    print(f"E6 saved: exp6_ctgan_segmentation.xlsx  ({len(synth)} rows)")
    return synth





# ── RUN ───────────────────────────────────────────────────────────────────
synthetic_data_e6 = generate_e6() # Directly call the function
print("\nFirst 5 rows of generated synthetic data (E6):")
display(synthetic_data_e6.head()) # Display the head of the DataFrame


Real data: (467, 33) | n_per: {0: 94, 1: 92, 2: 94, 3: 94, 4: 93}
E6 | Segment 0: fitting CTGAN on n=94...


/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:183: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


   Sampled 94 rows.
E6 | Segment 1: fitting CTGAN on n=92...


/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:183: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


   Sampled 92 rows.
E6 | Segment 2: fitting CTGAN on n=94...


...

/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:183: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


   Sampled 94 rows.
E6 | Segment 4: fitting CTGAN on n=93...


/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:183: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


   Sampled 93 rows.
E6 saved: exp6_ctgan_segmentation.xlsx  (467 rows)

First 5 rows of generated synthetic data (E6):


,No,q1,q2,q3,q4,q5,COL1,COL2,COL3,COL4,...,OT2,OT3,OT4,IB1,IB2,IB3,IB4,gender,age,occupation
0,10271070,1,1,1,4,1,4,4,5,5,...,4,4,3,4,4,3,4,1,4,2
1,11414878,1,1,1,2,1,4,4,4,5,...,4,3,3,4,4,3,4,1,3,4
2,10977807,1,1,1,4,5,5,3,2,4,...,3,3,3,4,4,5,4,1,4,4
3,5128785,1,1,1,4,2,4,5,5,5,...,4,4,4,4,4,2,3,2,1,5
4,10791753,1,1,1,3,4,5,5,4,4,...,4,4,4,4,3,5,4,1,1,2


In [6]:
from google.colab import files
files.download('exp6_ctgan_segmentation.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>